In [ ]:
# =============================================================================
# PROMPTS — defined first (edit here). All later cells use these names.
# =============================================================================
PROMPT = "A cinematic astronaut walking on Mars at sunrise, ultra detailed, realistic lighting, cinematic camera movement"
NEGATIVE_PROMPT = "blurry, distorted, low quality, static frame, artifacts"

# ZeroScope V2 576w — text-to-video (Kaggle GPU debug)

**Model:** **`cerspense/zeroscope_v2_576w`**, **fp16**, **576×320**. Optional enhancements use the **same `ENABLE_*` flags as `backend/config.py`** (see config cell — **defaults OFF** here for baseline-only runs). When enabled: cosine schedule once at scheduler init, gentle temporal blend (λ clamped 0.05–0.15), CLIP rerank after **full** runs (≤3 candidates). Motion diagnostic is **log-only**.

*(Notebook filename may still reference an older experiment — ignore.)*

**Run:** Settings → **GPU** → **Run All** once. For CLIP rerank: `pip install git+https://github.com/openai/CLIP.git` (optional).

## 1. Install dependencies

Single pip line for Kaggle (PyTorch + Diffusers + video utils).

In [ ]:
%pip install -q torch torchvision diffusers transformers accelerate imageio imageio-ffmpeg opencv-python Pillow scipy numpy
# Optional — only if ENABLE_CLIP_RERANK=True: pip install git+https://github.com/openai/CLIP.git

## 2. Imports

In [ ]:
from __future__ import annotations

import inspect
import os
import time
import warnings

import numpy as np
import torch
import torch.nn.functional as F
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Video, display
from PIL import Image

warnings.filterwarnings("ignore", category=UserWarning)

## 3. Config

Minimal generation settings for ZeroScope **576w** (native **576×320**).

In [ ]:
# --- Paths (Kaggle) ---
OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
VIDEO_BASENAME = "zeroscope_v2_576w_debug"
VIDEO_PATH = os.path.join(OUTPUT_DIR, f"{VIDEO_BASENAME}.mp4")

# --- ZeroScope V2 576w (Diffusers-compatible checkpoint) ---
MODEL_ID = "cerspense/zeroscope_v2_576w"

# --- Generation (minimal; matches model card orientation) ---
NUM_FRAMES = 24
NUM_INFERENCE_STEPS = 25
GUIDANCE_SCALE = 7.5
WIDTH = 576
HEIGHT = 320
FPS = 8

# Reproducibility
SEED = 42

# --- Enhancements (same flag names as backend/config.py; keep OFF for baseline on Kaggle) ---
ENABLE_COSINE_SCHEDULE = False
ENABLE_TEMPORAL_SMOOTHING = False
ENABLE_CLIP_RERANK = False
TEMPORAL_SMOOTHING_LAMBDA = 0.08
CLIP_NUM_CANDIDATES = 2

In [ ]:
# Mirror backend/enhancements.py — keep aligned when editing either file.
# Uses ENABLE_* / TEMPORAL_SMOOTHING_LAMBDA / CLIP_NUM_CANDIDATES from the config cell above.

import numpy as np
import torch
import torch.nn.functional as F
from diffusers import DPMSolverMultistepScheduler


def _clamp_lambda(v: float) -> float:
    return float(min(max(v, 0.05), 0.15))


def make_cosine_alphas_cumprod(num_steps: int, s: float = 0.008) -> torch.Tensor:
    steps = torch.arange(num_steps + 1, dtype=torch.float64)
    f = torch.cos(((steps / num_steps + s) / (1 + s)) * (np.pi / 2)) ** 2
    alphas_cumprod = f / f[0]
    alphas_cumprod = torch.clamp(alphas_cumprod, min=1e-5, max=1.0 - 1e-5)
    return alphas_cumprod[1:].float()


def build_scheduler(pipe):
    scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    if ENABLE_COSINE_SCHEDULE:
        n = len(scheduler.alphas_cumprod)
        cosine_ac = make_cosine_alphas_cumprod(n).to(
            device=scheduler.alphas_cumprod.device,
            dtype=scheduler.alphas_cumprod.dtype,
        )
        scheduler.alphas_cumprod = cosine_ac
        print("[enh] cosine alphas_cumprod applied (scheduler init only)")
    return scheduler


def blend_temporal_latents(lat: torch.Tensor) -> torch.Tensor:
    if not ENABLE_TEMPORAL_SMOOTHING:
        return lat
    if lat.dim() < 5 or lat.shape[2] <= 1:
        return lat
    w = _clamp_lambda(TEMPORAL_SMOOTHING_LAMBDA)
    out = lat.clone()
    prev = out[:, :, :-1]
    curr = out[:, :, 1:]
    out[:, :, 1:] = (1.0 - w) * curr + w * prev
    return torch.nan_to_num(out, nan=0.0, posinf=1e4, neginf=-1e4)


def clip_num_candidates() -> int:
    return max(1, min(int(CLIP_NUM_CANDIDATES), 3))


class CLIPReranker:
    def __init__(self, device: str):
        self.device = device
        self._model = None
        self._preprocess = None
        self._tried = False

    def _lazy_load(self):
        if self._model is not None or self._tried:
            return
        self._tried = True
        try:
            import clip

            self._model, self._preprocess = clip.load("ViT-B/32", device=self.device)
            self._model.eval()
            print("[enh] CLIP reranker loaded")
        except Exception as e:
            print(f"[enh] CLIP unavailable — rerank skipped ({e})")

    def score_frames(self, pil_frames, prompt: str) -> float:
        self._lazy_load()
        if self._model is None or not pil_frames:
            return 0.0
        try:
            import clip

            rgb = [f.convert("RGB") for f in pil_frames]
            idx = [
                max(0, len(rgb) // 4),
                len(rgb) // 2,
                min(len(rgb) - 1, 3 * len(rgb) // 4),
            ]
            text_tokens = clip.tokenize([prompt], truncate=True).to(self.device)
            with torch.no_grad():
                txt_feat = F.normalize(self._model.encode_text(text_tokens), dim=-1)
                sims = []
                for i in idx:
                    t = self._preprocess(rgb[i]).unsqueeze(0).to(self.device)
                    img_feat = F.normalize(self._model.encode_image(t), dim=-1)
                    sims.append((img_feat * txt_feat).sum().item())
            return float(np.mean(sims))
        except Exception as e:
            print(f"[enh] CLIP scoring failed: {e}")
            return 0.0

    def select_best(self, candidates, prompt: str):
        scores = [self.score_frames(c, prompt) for c in candidates]
        best_idx = int(np.argmax(scores)) if scores else 0
        print(f"[enh] CLIP rerank scores: {[f'{s:.4f}' for s in scores]} → pick [{best_idx}]")
        return candidates[best_idx], scores


## 4. GPU check

**CUDA + float16** for the pipeline weights.

In [ ]:
print("=" * 72)
print("[GPU] CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required. On Kaggle: Settings → Accelerator → GPU → re-run."
    )

DEVICE = "cuda"
DTYPE = torch.float16
print(f"[GPU] Device: {torch.cuda.get_device_name(0)}")
print(f"[GPU] Pipeline dtype: {DTYPE}")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"[GPU] Allocated (initial): {torch.cuda.memory_allocated() / 1e6:.1f} MB")
print("=" * 72)

## 5. Load model

- **`DiffusionPipeline.from_pretrained`** for ZeroScope  
- **`build_scheduler(pipe)`** → `DPMSolverMultistepScheduler`, optional cosine **once** at init if `ENABLE_COSINE_SCHEDULE`  
- **`enable_attention_slicing()`** for tighter VRAM  
- Optional latent temporal blend only when `ENABLE_TEMPORAL_SMOOTHING` (during denoise callbacks)

In [ ]:
print("[LOAD] Fetching ZeroScope V2 576w (first run downloads weights)...")
t_load = time.perf_counter()

pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
pipe.scheduler = build_scheduler(pipe)
pipe = pipe.to(DEVICE)

try:
    pipe.enable_attention_slicing()
    print("[LOAD] attention_slicing: enabled")
except Exception as e:
    print(f"[LOAD] attention_slicing skipped: {e}")

load_elapsed = time.perf_counter() - t_load
mem_mb = torch.cuda.memory_allocated() / 1e6
peak_mb = torch.cuda.max_memory_allocated() / 1e6
print("[LOAD] OK — model loaded successfully")
print(f"[LOAD] Pipeline: {pipe.__class__.__name__} | Scheduler: {pipe.scheduler.__class__.__name__}")
print(f"[LOAD] Load time: {load_elapsed:.1f}s")
print(f"[GPU] After load — allocated: {mem_mb:.1f} MB | peak: {peak_mb:.1f} MB")

## 6. Generate & export

Single or multi forward pass: **`ENABLE_CLIP_RERANK`** runs up to **3** full generations then CLIP-picks the best. Otherwise one **`generator`** run. Temporal blend uses **small** λ (0.05–0.15) with **`torch.nan_to_num`** only. Same flags as **`backend/config.py`** (defaults **OFF** above).

In [ ]:
def extract_frame_list(result) -> list:
    """Normalize Diffusers video output to a flat list of frames (PIL or array per frame)."""
    frames = result.frames
    if frames is None:
        raise RuntimeError("Pipeline returned no frames")
    first = frames[0]
    # Batch dimension: list of clips → take first clip
    if isinstance(first, (list, tuple)):
        return list(first)
    return list(frames)


def frames_to_pil_rgb(frames: list) -> list[Image.Image]:
    """Convert pipeline frames to RGB PIL for video writer (handles PIL or ndarray)."""
    out: list[Image.Image] = []
    for f in frames:
        if isinstance(f, Image.Image):
            out.append(f.convert("RGB"))
            continue
        arr = np.asarray(f)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.shape[-1] >= 3:
            arr = arr[..., :3]
        # Float pipelines often use [0,1] or [-1,1]
        if np.issubdtype(arr.dtype, np.floating):
            arr = arr.astype(np.float32)
            lo, hi = float(arr.min()), float(arr.max())
            if hi <= 1.01 and lo >= -1.01 and lo < 0:
                arr = (arr + 1.0) * 0.5
            if hi <= 1.01:
                arr = np.clip(arr, 0.0, 1.0) * 255.0
            else:
                arr = np.clip(arr, 0.0, 255.0)
            arr = np.round(arr).astype(np.uint8)
        else:
            arr = np.clip(arr, 0, 255).astype(np.uint8)
        out.append(Image.fromarray(arr, mode="RGB"))
    return out


def temporal_mean_abs_diff(pil_frames: list[Image.Image]) -> float:
    """Diagnostic: average |frame[t] - frame[t-1]| in [0,255] — near-zero suggests static/collapsed video."""
    if len(pil_frames) < 2:
        return 0.0
    diffs = []
    for a, b in zip(pil_frames[:-1], pil_frames[1:]):
        aa = np.asarray(a, dtype=np.float32)
        bb = np.asarray(b, dtype=np.float32)
        diffs.append(np.mean(np.abs(aa - bb)))
    return float(np.mean(diffs))


def run_inference(seed_run: int) -> list:
    torch.manual_seed(seed_run)
    np.random.seed(seed_run % (2**32))
    gen = torch.Generator(device=DEVICE).manual_seed(seed_run)
    pipe.scheduler = build_scheduler(pipe)

    call_kw = dict(
        prompt=PROMPT,
        num_frames=NUM_FRAMES,
        height=HEIGHT,
        width=WIDTH,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=gen,
    )

    sig = inspect.signature(pipe.__call__)

    def on_step_new(pipe, step_idx, timestep, callback_kwargs):
        if ENABLE_TEMPORAL_SMOOTHING and "latents" in callback_kwargs:
            callback_kwargs["latents"] = blend_temporal_latents(callback_kwargs["latents"])
        return callback_kwargs

    def on_step_legacy(step_idx, timestep, latents_tensor):
        if ENABLE_TEMPORAL_SMOOTHING:
            blended = blend_temporal_latents(latents_tensor)
            latents_tensor.copy_(blended)

    if ENABLE_TEMPORAL_SMOOTHING:
        if "callback_on_step_end" in sig.parameters:
            call_kw["callback_on_step_end"] = on_step_new
            call_kw["callback_on_step_end_tensor_inputs"] = ["latents"]
        else:
            call_kw["callback"] = on_step_legacy
            call_kw["callback_steps"] = 1

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        try:
            out = pipe(negative_prompt=NEGATIVE_PROMPT, **call_kw)
        except TypeError:
            out = pipe(**call_kw)

    raw_frames = extract_frame_list(out)
    return frames_to_pil_rgb(raw_frames)


torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()

clip_note = None
if ENABLE_CLIP_RERANK and clip_num_candidates() >= 2:
    n = clip_num_candidates()
    runs = []
    for r in range(n):
        print(f"[GEN] candidate {r + 1}/{n}...")
        runs.append(run_inference(SEED + r))
    reranker = CLIPReranker(DEVICE)
    pil_frames, clip_scores = reranker.select_best(runs, PROMPT)
    clip_note = max(clip_scores) if clip_scores else None
else:
    pil_frames = run_inference(SEED)

generation_seconds = time.perf_counter() - t0
peak_after_mb = torch.cuda.max_memory_allocated() / 1e6

n_export = len(pil_frames)
motion_score = temporal_mean_abs_diff(pil_frames)

print("\n" + "=" * 72)
print("[GEN] Complete")
print(f"[GEN] Wall time (generation): {generation_seconds:.2f}s")
print(f"[GEN] Frames returned: {n_export} (requested NUM_FRAMES={NUM_FRAMES})")
print(f"[GEN] FPS (export): {FPS}")
print(f"[GEN] Mean |Δframe| (diagnostic motion): {motion_score:.3f}")
if clip_note is not None:
    print(f"[GEN] CLIP rerank (max score): {clip_note:.4f}")
if motion_score < 0.5 and n_export > 1:
    print("[GEN] WARN: very low inter-frame delta — check model/output if video looks static")
print(f"[GPU] Peak memory (generation): {peak_after_mb:.1f} MB")
print("=" * 72)

## 7. Save & display

Write **MP4** under **`/kaggle/working/outputs/`** and show a short debug summary.

In [ ]:
print(f"[SAVE] Writing MP4 → {VIDEO_PATH}")
export_to_video(pil_frames, VIDEO_PATH, fps=FPS)

print("\n" + "=" * 72)
print("SUMMARY")
print("=" * 72)
print(f"Model:             {MODEL_ID}")
print(f"Resolution:        {WIDTH}x{HEIGHT}")
print(f"Frames / FPS:      {n_export} @ {FPS} fps")
print(f"Generation time:   {generation_seconds:.2f}s")
print(f"Seed:              {SEED}")
print(f"GPU peak memory:   {peak_after_mb:.1f} MB")
print(f"Output:            {VIDEO_PATH}")
print("=" * 72)

display(Video(VIDEO_PATH, embed=True))